# Análise de sensibilidade dos parâmetros do pipeline

Avalia, no conjunto de validação, a sensibilidade do pipeline à alteração de
um parâmetro por vez (OFAT). As saídas principais são o Dice da segmentação e
o sucesso dos óstios, considerando casos corretos ou toleráveis. O conjunto
de teste não participa da seleção dos parâmetros.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT_CANDIDATES = (CURRENT_DIR, *CURRENT_DIR.parents)
REPO_ROOT = next(
    root for root in REPO_ROOT_CANDIDATES if (root / "src" / "utils").is_dir()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    build_parameter_pairwise_summary,
    build_parameter_sensitivity_summary,
    build_threshold_performance_data,
    compute_effective_upper_thresholds,
)
from utils.project.notebook_env import resolve_imagecas_base_path  # noqa: E402

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

## Configuração

Por padrão, o notebook abre o run completo indicado abaixo. O cálculo dos
thresholds efetivos em HU pode ser desligado para uma execução mais rápida.

In [ ]:
# Define os runs quantitativos analisados.
ANALYSIS_DIR = REPO_ROOT / "output/segmentation/analysis/pipeline_parameter_validation"
RUNS_DIR = ANALYSIS_DIR / "runs"
RUN_NAME = "sensitivity_cbeb_p999_val_270"
THRESHOLD_RUN_NAME = "sensitivity_selected_val_270"
THRESHOLD_DELTA_TOP_N = 30
COMPUTE_THRESHOLD_HU = True

try:
    IMAGECAS_PATH = resolve_imagecas_base_path()
except FileNotFoundError:
    IMAGECAS_PATH = None

In [ ]:
def find_run_dir(run_name=None):
    if run_name:
        candidate = RUNS_DIR / run_name
        if not candidate.is_dir():
            raise FileNotFoundError(f"Run não encontrado: {candidate}")
        return candidate

    candidates = [
        path
        for path in RUNS_DIR.glob("*")
        if (path / "summary/ranking.csv").is_file()
        and (path / "results/image_results.csv").is_file()
    ]
    if not candidates:
        raise FileNotFoundError(
            "Nenhum run completo encontrado. Execute o experimento de sensibilidade."
        )
    return max(candidates, key=lambda path: path.stat().st_mtime)


def load_run(run_name):
    run_dir = find_run_dir(run_name)
    results = pd.read_csv(run_dir / "results/image_results.csv")
    parameters = pd.read_csv(run_dir / "parameters/variant_parameters.csv")
    config = json.loads((run_dir / "run_config.json").read_text(encoding="utf-8"))
    if config.get("split") != "val" or set(results["split"].dropna()) != {"val"}:
        raise ValueError(f"O run {run_name} não contém apenas o split de validação.")
    return run_dir, results, parameters, config


# Carrega o run P99.9 e o run histórico que contém P99.7/P99.5.
RUN_DIR, primary_results, primary_parameters, run_config = load_run(RUN_NAME)
THRESHOLD_RUN_DIR, threshold_results, threshold_parameters, threshold_run_config = load_run(
    THRESHOLD_RUN_NAME
)

# O baseline histórico representa P99.7; P99.9 já é o baseline do run principal.
threshold_aliases = {"baseline": "upper_p997", "upper_p995": "upper_p995"}
threshold_results = threshold_results.loc[
    threshold_results["variant"].isin(threshold_aliases)
].copy()
threshold_results["variant"] = threshold_results["variant"].replace(threshold_aliases)

threshold_parameters = threshold_parameters.loc[
    threshold_parameters["variant"].isin(threshold_aliases)
].copy()
threshold_parameters["variant"] = threshold_parameters["variant"].replace(threshold_aliases)
threshold_parameters.loc[
    threshold_parameters["variant"] == "upper_p997", "description"
] = "Percentil superior do threshold = 99.7."

results_df = pd.concat([primary_results, threshold_results], ignore_index=True)
parameters_df = pd.concat([primary_parameters, threshold_parameters], ignore_index=True)

results_df.drop(columns=['cohort_kind', 'cohort_roles'], inplace=True)

primary_ids = set(primary_results["IMG_ID"])
threshold_ids = set(threshold_results["IMG_ID"])
if primary_ids != threshold_ids:
    raise ValueError("Os runs P99.9, P99.7 e P99.5 não usam os mesmos exames.")

print(f"Run principal: {RUN_DIR.relative_to(REPO_ROOT)}")
print(f"Run dos thresholds: {THRESHOLD_RUN_DIR.relative_to(REPO_ROOT)}")
print(f"Imagens: {results_df['IMG_ID'].nunique()} | Variantes: {results_df['variant'].nunique()}")

results_df.head()

## Parâmetros avaliados

A referência é lida do snapshot do próprio run. A tabela mostra somente os
quatro parâmetros alterados na análise OFAT.

In [ ]:
# Usa o snapshot do run para documentar os valores realmente avaliados.
reference_variant = next(item for item in run_config["variants"] if item["name"] == "baseline")
reference = reference_variant["overrides"]

baseline_values = pd.DataFrame(
    [
        ("Upper percentile", reference["MAX_THRESHOLD_PERCENTILE"], "99.5, 99.7, 99.9"),
        (
            "Limite z do segundo óstio (mm)",
            reference["OSTIA_DETECTION.max_z_diff_mm"],
            "30, 40, 50",
        ),
        (
            "Divisor global do RG",
            reference["REGION_GROWING.threshold_divisor"],
            "5, 7, 9",
        ),
        (
            "Fração mínima de vesselness",
            reference["REGION_GROWING.min_vesselness_fraction"],
            "0.05, 0.078, 0.09",
        ),
    ],
    columns=["parâmetro", "referência canônica", "níveis avaliados"],
)
baseline_values

## Resultados quantitativos

A tabela abaixo contém diretamente as medidas relevantes para a análise de
sensibilidade: número e taxa de sucessos dos óstios, além de média, desvio
padrão e mediana do Dice. O sucesso aceita localizações
classificadas como corretas ou toleráveis.

In [ ]:
# Resume Dice e sucesso dos óstios para cada configuração.
sensitivity_summary = build_parameter_sensitivity_summary(
    results_df, parameters_df, baseline_variant="baseline"
)
article_sensitivity_table = sensitivity_summary[[
    "variant", "description", "images",
    "ostia_success_count", "ostia_success_percent",
    "mean_dice", "std_dice", "median_dice",
    "delta_dice_vs_baseline",
]]
article_sensitivity_table.round(4)

### Dice médio e variabilidade por configuração

In [ ]:
# Ordena as variantes antes de desenhar o Dice médio.
plot_df = sensitivity_summary.sort_values("mean_dice", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    plot_df["variant"],
    plot_df["mean_dice"],
    xerr=plot_df["std_dice"].fillna(0),
    color="#2878B5",
    alpha=0.9,
    capsize=3,
)
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
ax.set_xlabel("Dice Score médio", fontsize=13)
ax.set_ylabel("Configuração", fontsize=13)
ax.set_xlim(0, 1)
fig.tight_layout()
plt.show()

### Taxa de sucesso na localização dos óstios

In [ ]:
# Mostra a sensibilidade da localização dos óstios aos parâmetros.
plot_df = sensitivity_summary.sort_values("ostia_success_percent", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    plot_df["variant"],
    plot_df["ostia_success_percent"],
    color="#3A923A",
    alpha=0.9,
)
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=10)
ax.set_xlabel("Óstios corretos ou toleráveis (%)", fontsize=13)
ax.set_ylabel("Configuração", fontsize=13)
ax.set_xlim(0, 100)
fig.tight_layout()
plt.show()

## Alterações pareadas em relação à referência

Como todas as configurações usam os mesmos IDs, a diferença de Dice é calculada
exame a exame. Essa tabela ajuda a distinguir uma mudança sistemática de uma
média dominada por poucos casos.

In [ ]:
# Compara cada variante com o baseline nos mesmos exames.
pairwise_df = build_parameter_pairwise_summary(
    results_df, baseline_variant="baseline"
)
pairwise_df.round(4)

## Comparação dos thresholds nos casos de maior variação

Esta seção usa P99.9 como baseline e, para cada exame, escolhe a melhor
alternativa entre P99.7 e P99.5. A análise detalhada considera os exames com
maior variação absoluta de Dice, independentemente de melhora ou piora.

### 1. Desempenho global por percentil

Compara P99.9, P99.7 e P99.5 considerando todos os 270 exames. A tabela mostra
Dice médio e mediano, sucesso dos óstios e a fração média do volume preservada
pelo threshold. P99.9 é a configuração de referência.

In [ ]:
THRESHOLD_VARIANTS = ("baseline", "upper_p997", "upper_p995")
THRESHOLD_LABELS = {
    "baseline": "P99.9 (baseline)",
    "upper_p997": "P99.7",
    "upper_p995": "P99.5",
}

# Organiza os três percentis usando os mesmos exames de validação.
threshold_performance_df = build_threshold_performance_data(
    results_df,
    parameters_df,
    variants=THRESHOLD_VARIANTS,
)
overall_threshold_summary = (
    threshold_performance_df.groupby(["variant", "upper_percentile"], as_index=False)
    .agg(
        images=("IMG_ID", "nunique"),
        mean_dice=("dice_artery", "mean"),
        median_dice=("dice_artery", "median"),
        ostia_success_rate=("ostia_success", "mean"),
        mean_threshold_volume_fraction=("threshold_volume_fraction", "mean"),
    )
)
overall_threshold_summary["ostia_success_percent"] = (
    100 * overall_threshold_summary["ostia_success_rate"]
)
overall_threshold_summary = overall_threshold_summary.sort_values(
    "upper_percentile", ascending=False
)
overall_threshold_summary["variant"] = overall_threshold_summary["variant"].map(
    THRESHOLD_LABELS
)
display(
    overall_threshold_summary.drop(columns="ostia_success_rate")
    .reset_index(drop=True)
    .round(4)
)

### 2. Seleção dos exames mais sensíveis

Para cada exame, P99.9 é comparado com a melhor alternativa entre P99.7 e
P99.5. Em seguida, são selecionados os exames com maior variação absoluta de
Dice. Portanto, entram tanto grandes melhorias quanto grandes pioras.

In [ ]:
# Compara o baseline com a melhor alternativa individual de cada exame.
threshold_dice_wide = threshold_performance_df.pivot(
    index="IMG_ID", columns="variant", values="dice_artery"
)
alternative_order = ["upper_p997", "upper_p995"]
alternative_dice = threshold_dice_wide[alternative_order]

threshold_delta_df = pd.DataFrame(index=threshold_dice_wide.index)
threshold_delta_df["baseline_dice"] = threshold_dice_wide["baseline"]
threshold_delta_df["best_threshold_variant"] = alternative_dice.idxmax(axis=1)
threshold_delta_df["best_threshold_dice"] = alternative_dice.max(axis=1)
threshold_delta_df["delta_dice"] = (
    threshold_delta_df["best_threshold_dice"] - threshold_delta_df["baseline_dice"]
)
threshold_delta_df["absolute_delta_dice"] = threshold_delta_df["delta_dice"].abs()
threshold_delta_df = threshold_delta_df.reset_index()

# Seleciona mudanças grandes em ambas as direções, não apenas melhorias.
largest_threshold_changes = threshold_delta_df.nlargest(
    THRESHOLD_DELTA_TOP_N, "absolute_delta_dice"
).copy()
largest_threshold_changes[
    ["IMG_ID", "baseline_dice", "best_threshold_variant", "best_threshold_dice", "delta_dice"]
].round(4)

### 3. Threshold efetivo em HU

O percentil configurado não corresponde a um valor HU fixo. Esta etapa carrega
somente os exames selecionados, aplica o mesmo downsampling do pipeline e
calcula os valores efetivos de P99.5, P99.7 e P99.9 em HU.

In [ ]:
# Calcula os valores HU somente para os exames de maior variação.
effective_thresholds_df = pd.DataFrame()
if COMPUTE_THRESHOLD_HU:
    if IMAGECAS_PATH is None:
        print("Defina IMAGECAS_BASE_PATH para calcular os thresholds efetivos em HU.")
    else:
        effective_thresholds_df = compute_effective_upper_thresholds(
            largest_threshold_changes["IMG_ID"].astype(int).tolist(),
            IMAGECAS_PATH,
            run_config["effective_base_config"],
            percentiles=(99.5, 99.7, 99.9),
        )
else:
    print("Cálculo em HU desativado por COMPUTE_THRESHOLD_HU = False.")

### 4. Baseline versus melhor threshold por exame

A tabela reúne o Dice e o threshold HU do baseline, a melhor alternativa do
mesmo exame e o respectivo delta. Delta positivo significa melhora sobre P99.9;
delta negativo significa que reduzir o percentil piorou o resultado.

In [ ]:
# Monta uma linha por exame com baseline, melhor alternativa e delta.
if effective_thresholds_df.empty:
    threshold_delta_details = largest_threshold_changes.copy()
    print("Ative COMPUTE_THRESHOLD_HU para incluir os thresholds efetivos.")
else:
    thresholds_hu_wide = effective_thresholds_df.pivot(
        index="IMG_ID", columns="upper_percentile", values="max_threshold_hu"
    )
    threshold_delta_details = largest_threshold_changes.merge(
        thresholds_hu_wide,
        left_on="IMG_ID",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
    threshold_delta_details["baseline_threshold_hu"] = threshold_delta_details[99.9]
    threshold_delta_details["best_threshold_hu"] = threshold_delta_details.apply(
        lambda row: row[99.7]
        if row["best_threshold_variant"] == "upper_p997"
        else row[99.5],
        axis=1,
    )

threshold_delta_details["best_threshold"] = threshold_delta_details[
    "best_threshold_variant"
].map(THRESHOLD_LABELS)
threshold_delta_details["result"] = "Empate"
threshold_delta_details.loc[threshold_delta_details["delta_dice"] > 0, "result"] = "Melhorou"
threshold_delta_details.loc[threshold_delta_details["delta_dice"] < 0, "result"] = "Piorou"

comparison_columns = [
    "IMG_ID", "baseline_dice", "baseline_threshold_hu", "best_threshold",
    "best_threshold_dice", "best_threshold_hu", "delta_dice", "result",
]
display(
    threshold_delta_details.reindex(columns=comparison_columns)
    .sort_values("delta_dice", ascending=False)
    .reset_index(drop=True)
    .round(3)
)

### 5. Configuração com maior Dice por exame

Conta quantos exames foram vencidos isoladamente por cada percentil. Quando
duas ou três configurações atingem exatamente o mesmo maior Dice, o exame é
registrado como empate.

In [ ]:
# Resume qual percentil alcança o maior Dice em cada exame, preservando empates.
maximum_dice = threshold_dice_wide.max(axis=1)
number_of_best_variants = threshold_dice_wide.eq(maximum_dice, axis=0).sum(axis=1)
best_variant_by_image = threshold_dice_wide.idxmax(axis=1).where(
    number_of_best_variants.eq(1), "tie"
)
best_threshold_counts = (
    best_variant_by_image.value_counts()
    .rename_axis("best_threshold")
    .reset_index(name="images")
)
best_threshold_counts["best_threshold"] = best_threshold_counts["best_threshold"].replace(
    {**THRESHOLD_LABELS, "tie": "Empate"}
)
best_threshold_counts["percent"] = 100 * best_threshold_counts["images"] / len(
    best_variant_by_image
)
best_threshold_counts.round(2)

### 6. Visualização das maiores variações

O gráfico à esquerda mostra o delta de Dice dos exames mais sensíveis. O
gráfico à direita liga o threshold HU do baseline ao threshold HU da melhor
alternativa, permitindo observar a intensidade da mudança aplicada.

In [ ]:
# Destaca visualmente as maiores melhorias e pioras em relação ao baseline.
plot_delta = threshold_delta_details.sort_values("delta_dice").copy()
colors = plot_delta["delta_dice"].map(
    lambda value: "#2a9d8f" if value > 0 else "#d1495b"
)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

axes[0].barh(plot_delta["IMG_ID"].astype(str), plot_delta["delta_dice"], color=colors)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Delta Dice (melhor threshold - baseline)", fontsize=12)
axes[0].set_ylabel("Exame", fontsize=12)

if not effective_thresholds_df.empty:
    positions = range(len(plot_delta))
    axes[1].scatter(
        plot_delta["baseline_threshold_hu"], positions,
        color="#1f77b4", label="P99.9 baseline", s=38,
    )
    axes[1].scatter(
        plot_delta["best_threshold_hu"], positions,
        color="#e76f51", label="Melhor alternativa", s=38,
    )
    for position, (_, row) in zip(positions, plot_delta.iterrows()):
        axes[1].plot(
            [row["baseline_threshold_hu"], row["best_threshold_hu"]],
            [position, position], color="#888888", linewidth=0.8,
        )
    axes[1].set_yticks(list(positions), plot_delta["IMG_ID"].astype(str))
    axes[1].set_xlabel("Threshold superior efetivo (HU)", fontsize=12)
    axes[1].set_ylabel("Exame", fontsize=12)
    axes[1].legend()
else:
    axes[1].set_visible(False)

plt.show()

### 7. Melhor threshold dentro da faixa de intensidades da imagem

Para os exames com maior variação de Dice, esta tabela identifica qual entre
P99.9, P99.7 e P99.5 produziu o maior resultado. O threshold efetivo é situado
entre a menor e a maior intensidade do volume **após o mesmo downsampling do
pipeline**. A distância até o máximo ajuda a verificar se o percentil está
apenas removendo valores extremos ou recortando uma parcela mais ampla da faixa
superior de intensidades.


In [ ]:
# Relaciona o threshold vencedor à faixa HU real de cada volume analisado.
if effective_thresholds_df.empty:
    threshold_intensity_analysis = pd.DataFrame()
    print(
        "Ative COMPUTE_THRESHOLD_HU para comparar o melhor threshold "
        "com as intensidades mínima e máxima."
    )
else:
    selected_ids = largest_threshold_changes["IMG_ID"].astype(int)
    selected_dice = threshold_dice_wide.loc[selected_ids]

    # Identifica o percentil com maior Dice, incluindo o baseline P99.9.
    best_dice = selected_dice.max(axis=1)
    winner_count = selected_dice.eq(best_dice, axis=0).sum(axis=1)
    winner_variant = selected_dice.idxmax(axis=1).where(winner_count.eq(1), "tie")
    winner_percentile = winner_variant.map(
        {"baseline": 99.9, "upper_p997": 99.7, "upper_p995": 99.5}
    )

    threshold_intensity_analysis = pd.DataFrame(
        {
            "IMG_ID": selected_dice.index.astype(int),
            "best_threshold_variant": winner_variant.values,
            "best_upper_percentile": winner_percentile.values,
            "best_dice": best_dice.values,
        }
    )

    # Cada exame repete os mesmos limites nas três linhas de percentil.
    intensity_bounds = (
        effective_thresholds_df.groupby("IMG_ID", as_index=False)
        .agg(
            image_min_hu=("image_min_hu", "first"),
            image_max_hu=("image_max_hu", "first"),
            image_intensity_range_hu=("image_intensity_range_hu", "first"),
        )
    )
    threshold_intensity_analysis = threshold_intensity_analysis.merge(
        intensity_bounds,
        on="IMG_ID",
        how="left",
        validate="one_to_one",
    )

    # Anexa o valor HU correspondente ao percentil vencedor de cada exame.
    winner_thresholds = effective_thresholds_df.rename(
        columns={
            "upper_percentile": "best_upper_percentile",
            "max_threshold_hu": "best_threshold_hu",
        }
    )[["IMG_ID", "best_upper_percentile", "best_threshold_hu"]]
    threshold_intensity_analysis = threshold_intensity_analysis.merge(
        winner_thresholds,
        on=["IMG_ID", "best_upper_percentile"],
        how="left",
        validate="one_to_one",
    )

    threshold_intensity_analysis["distance_threshold_to_max_hu"] = (
        threshold_intensity_analysis["image_max_hu"]
        - threshold_intensity_analysis["best_threshold_hu"]
    )
    threshold_intensity_analysis["threshold_position_percent"] = 100 * (
        threshold_intensity_analysis["best_threshold_hu"]
        - threshold_intensity_analysis["image_min_hu"]
    ) / threshold_intensity_analysis["image_intensity_range_hu"]
    threshold_intensity_analysis["best_threshold"] = (
        threshold_intensity_analysis["best_threshold_variant"]
        .replace({**THRESHOLD_LABELS, "tie": "Empate"})
    )

    analysis_columns = [
        "IMG_ID",
        "image_min_hu",
        "image_max_hu",
        "best_threshold",
        "best_threshold_hu",
        "distance_threshold_to_max_hu",
        "threshold_position_percent",
        "best_dice",
    ]
    display(
        threshold_intensity_analysis[analysis_columns]
        .sort_values("best_dice", ascending=False)
        .reset_index(drop=True)
        .round(3)
    )

    print(
        "Posição mediana do melhor threshold na faixa HU: "
        f"{threshold_intensity_analysis['threshold_position_percent'].median():.2f}%"
    )
    print(
        "Distância mediana entre o melhor threshold e a intensidade máxima: "
        f"{threshold_intensity_analysis['distance_threshold_to_max_hu'].median():.1f} HU"
    )
